# Modelar la Volatilidad, GARCH(p,q)

En este ejercicio de aplicación ilustramos el ajuste de un modelo GARCH(p,q) para estimar la volatilidad condicional. Estos resultados se usan para el ejemplo de aplicación en las notas de clase

## Precios del petróleo, Brent

A continuación modelaremos la serie de precio del petróleo en su referencia Brent. 

In [ ]:
paquetes <- c("quantmod", "rugarch", "tseries", "FinTS")
faltantes <- paquetes[!(paquetes %in% installed.packages()[, "Package"])]
if (length(faltantes) > 0) install.packages(faltantes)

invisible(lapply(paquetes, library, character.only = TRUE))

In [ ]:
# ---------------------------------------------------------------------------
# DESCARGA DE DATOS DESDE YAHOO FINANCE
# ---------------------------------------------------------------------------
# Ticker del petróleo Brent (ICE Brent Crude Oil futures) en Yahoo Finance: "BZ=F"

ticker    <- "BZ=F"
fecha_fin <- Sys.Date()
fecha_ini <- fecha_fin - 5 * 365  # aprox. 5 años de historia

brent <- getSymbols(
  Symbols     = ticker,
  src         = "yahoo",
  from        = fecha_ini,
  to          = fecha_fin,
  auto.assign = FALSE
)

# Nos quedamos con el precio de cierre y eliminamos NAs (días sin cotización)
precio <- na.omit(Cl(brent))
colnames(precio) <- "Cierre"


Graficamos y analizamos

In [ ]:
plot(precio, main = "Precio de cierre - Petróleo Brent (BZ=F)", col = "steelblue")

Calculamos los retornos logarítmicos diarios y analizamos sus propiedades y comportamiento

In [ ]:
retornos <- na.omit(diff(log(precio)) * 100)
colnames(retornos) <- "retorno"

plot(retornos, main = "Retornos diarios - Brent", col = "darkred")
hist(retornos, breaks = 60, main = "Distribución de los retornos", col = "gray80",
     xlab = "Retorno (%)")

# Estadísticos descriptivos
summary(retornos)
cat("SD:",sd(retornos),"\n")
skewness <- function(x) {
  x <- x - mean(x)
  (mean(x^3)) / (mean(x^2))^1.5
}
cat("Asimetría:", skewness(as.numeric(retornos)), "\n")
moments::kurtosis(retornos)

# Estacionariedad de la serie de retornos (Dickey-Fuller aumentado)
adf.test(retornos)

# Autocorrelación de los retornos y de los retornos al cuadrado
par(mfrow = c(2, 2))
acf(retornos,      main = "ACF retornos")
pacf(retornos,     main = "PACF retornos")
acf(retornos^2,    main = "ACF retornos^2")
pacf(retornos^2,   main = "PACF retornos^2")
par(mfrow = c(1, 1))


Dado que la serie cumple con las condiciones de estacionariedad y dependencia débil, y hay evidencia de dependencia de los retornos al cuadrado, hacemos las pruebas formales de efectos ARCH

In [ ]:
# Ljung-Box sobre los retornos al cuadrado: H0 = no hay autocorrelación
# (si se rechaza, hay evidencia de heterocedasticidad condicional -> ARCH)
Box.test(retornos^2, lag = 12, type = "Ljung-Box")

# Prueba ARCH-LM de Engle
FinTS::ArchTest(retornos, lags = 12)

Los resulatdos justifican el uso de un modelo de volatilidad condicional. 

Usamos un ARMA(0,0) para la media y GARCH(1,1) para la varianza. Dado que los retornos logarítimicos diarios no exhiben correlación serial, es razonable esta especificación. Para la varianza, esta especificación es estándar, y si las pruebas sobre los residuales son satisfactorias entonces nos quedamos con este modelo. Dada la evidencia de colas gordas, usamos la distribución de t de student 

In [ ]:
spec_garch <- ugarchspec(
  variance.model = list(model = "sGARCH", garchOrder = c(1, 1)),
  mean.model     = list(armaOrder = c(0, 0), include.mean = TRUE),
  distribution.model = "std"
)

fit_garch <- ugarchfit(spec = spec_garch, data = retornos, solver = "hybrid")

show(fit_garch)          # resumen completo: coeficientes, criterios de info, pruebas
coef(fit_garch)          # coeficientes estimados
infocriteria(fit_garch)  # AIC, BIC, etc.

coef(fit_garch)["alpha1"]+coef(fit_garch)["beta1"]
coef(fit_garch)["omega"]/(1-(coef(fit_garch)["alpha1"]+coef(fit_garch)["beta1"]))


Los resultados muestran una alta persistencia de la varianza $\hat{\alpha}_1+\hat{\beta}_1=0.984$. Así mismo, la varianza de largo plazo es $\hat{\sigma}^2=\dfrac{\hat{\omega}}{1-(\hat{\alpha}_1+\hat{\beta}_1)}=2.67$

In [ ]:
# Se evalúa si, tras estimar el GARCH, los residuales estandarizados se
# comportan como ruido blanco y si desaparece la heterocedasticidad
# condicional (si no desaparece, el modelo no capturó bien la dinámica).

res_std <- residuals(fit_garch, standardize = TRUE)

# Autocorrelación remanente en los residuales estandarizados
Box.test(res_std,    lag = 10, type = "Ljung-Box")
# Autocorrelación remanente en los residuales al cuadrado (efectos ARCH residuales)
Box.test(res_std^2,  lag = 10, type = "Ljung-Box")
# Prueba ARCH-LM sobre los residuales estandarizados
FinTS::ArchTest(res_std, lags = 10)

# Normalidad / bondad de ajuste de la distribución supuesta
jarque.bera.test(as.numeric(res_std))
qqnorm(as.numeric(res_std), main = "QQ-plot de residuales estandarizados")
qqline(as.numeric(res_std), col = "red")

# Prueba de sesgo de signo (Engle-Ng): detecta asimetrías/efecto apalancamiento
#     no capturadas por un GARCH simétrico
signbias(fit_garch)

# Estabilidad de los parámetros en el tiempo (test de Nyblom)
nyblom(fit_garch)


Los resultados sobre los residuales son satisfactorios en la medida que los residuales no muestran correlación serial. La prueba de sign-bias sugiere que los choques son asimétricos, siendo más importantes las caídas de precio que las subidas. Finalmente, la prueba de estabilidad no rechaza la hipótesis nula de estabilidad de los parámetros

Graficamos la volatilidad

In [ ]:
vol<-sigma(fit_garch)
plot(vol)

Ahora hacemos el pronóstico de la volatilidad

In [ ]:
pronostico <- ugarchforecast(fit_garch, n.ahead = 30)
pronostico
plot(pronostico, which = 1)  # pronóstico de la serie (retornos)
plot(pronostico, which = 3)  # pronóstico de la volatilidad (sigma) condicional

## Actividad

Usando datos diarios de la tasa de cambio USDCOP, estime un modelo de volatilidad. Investigue como usaría ese resultado para cubrir el riesgo de apreciación del peso de una empresa exportadora que tiene una cuenta por cobrar de USD 3 millones a 60 días. El instrumento de cobertura es un forward USDCOP